## 1. Vérifier le GPU

In [ ]:
!nvidia-smi
import torch
print("CUDA disponible :", torch.cuda.is_available())

## 2. Installer les dépendances

In [ ]:
!pip install -q ultralytics roboflow

## 3. Télécharger le dataset depuis Roboflow

In [ ]:
ROBOFLOW_API_KEY = "comEqKvr0ukmsyevFUqM"
WORKSPACE = "graduation-project-2023"
PROJECT = "plants-diseases-detection-and-classification"
VERSION = 12

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

print("Dataset téléchargé dans :", dataset.location)
!cat {dataset.location}/data.yaml

## 4. Entraîner YOLOv8 (détection)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,   # early stopping si pas d'amélioration
    name='crop_disease_run'
)

## 5. Évaluer le modèle (précision, rappel, mAP)

In [ ]:
metrics = model.val()
print("mAP50-95 :", metrics.box.map)
print("mAP50    :", metrics.box.map50)
print("Précision:", metrics.box.mp)
print("Rappel   :", metrics.box.mr)

## 6. Tester sur une image (avec boîtes dessinées)

In [ ]:
from google.colab import files

uploaded = files.upload()
test_image = list(uploaded.keys())[0]

results = model.predict(source=test_image, save=True, conf=0.25)

# Affiche l'image annotée
import glob
from IPython.display import Image as IPImage, display

latest_pred = sorted(glob.glob('runs/detect/predict*/*.jpg'))[-1]
display(IPImage(filename=latest_pred))

for r in results:
    for box in r.boxes:
        cls_name = model.names[int(box.cls)]
        conf = float(box.conf)
        print(f"{cls_name} — {conf*100:.1f}%")

## 7. Télécharger les poids entraînés

In [ ]:
import shutil
from google.colab import files
shutil.copy('runs/detect/crop_disease_run/weights/best.pt', 'best_cropdisease.pt')

files.download('best_cropdisease.pt')